In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f
import pandas as pd
from functools import reduce
from pathlib import Path
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("GenPM")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

In [ ]:
# This notebook is for creating eda_preprocessed dataframes

In [ ]:
data_paths = {
    "pm_raw": [
        "/workspaces/GenPM---temporary/data/shared_dir/raw_data/pm_kpis_part1.parquet",
        "/workspaces/GenPM---temporary/data/shared_dir/raw_data/pm_kpis_part2.parquet",
        "/workspaces/GenPM---temporary/data/shared_dir/raw_data/pm_kpis_part3.parquet",
        "/workspaces/GenPM---temporary/data/shared_dir/raw_data/pm_kpis_part4.parquet",
        "/workspaces/GenPM---temporary/data/shared_dir/raw_data/pm_kpis_part5.parquet"
    ],
    "kpis_definitions": "/workspaces/GenPM---temporary/data/shared_dir/raw_data/kpis_definitions.parquet",
    "simple_reports": "/workspaces/GenPM---temporary/data/shared_dir/raw_data/simple_reports.parquet"
}

In [ ]:
list_of_pm_dfs = [spark.read.parquet(dp) for dp in data_paths["pm_raw"]]
kpis_definitions_df = spark.read.parquet(data_paths["kpis_definitions"])
simple_reports_df = spark.read.parquet(data_paths["simple_reports"])

In [ ]:
pm_df_all: DataFrame = reduce(lambda df1, df2: df1.unionByName(df2), list_of_pm_dfs)
pm_df_all.printSchema()

In [ ]:
eda_data_path = Path("/workspaces/GenPM---temporary/data/shared_dir/eda_data")
raw_pm_path = eda_data_path / "raw_pm_data"
sample_path = eda_data_path / "sample"
pm_stats_path = eda_data_path / "stats"
pm_agg_path = eda_data_path / "agg"
other_data = eda_data_path / "pm_metadata"

In [ ]:
pm_df_all = pm_df_all.withColumn("start_date", f.to_date("start_time"))
pm_df_all = pm_df_all.withColumn("start_hour", f.hour("start_time"))
pm_df_all = pm_df_all.withColumnsRenamed({
    "bts_anon": "bts_id",
    "distname_anon": "distname"
})

In [ ]:
# simple rewrites
pm_df_all.write.partitionBy("start_date").parquet(str(raw_pm_path), mode="overwrite")
kpis_definitions_df.write.parquet(str(other_data / "kpis_definitions"), mode="overwrite")
simple_reports_df.write.parquet(str(other_data / "simple_reports"), mode="overwrite")

In [ ]:
# calculate stats df
pm_df_stats = pm_df_all.groupBy("kpi_id").agg(
    f.count("*").alias("count"),
    f.avg("kpi_value").alias("mean"),
    f.stddev("kpi_value").alias("std"),
    f.min("kpi_value").alias("min"),
    f.max("kpi_value").alias("max"),
    f.expr("percentile_approx(kpi_value, array(0.25, 0.5, 0.75))").alias("quantiles")
)

# pm aggregated per day
pm_df_grouped_date = pm_df_all.groupBy("bts_id", "distname", "kpi_id", "start_date").agg(
    f.avg("kpi_value").alias("kpi_mean"),
    f.min("kpi_value").alias("kpi_min"),
    f.max("kpi_value").alias("kpi_max"),
    f.count("*").alias("kpi_count"),
    # add other aggregations in need
)

# pm aggregated per hour
pm_df_grouped_hours = pm_df_all.groupBy("bts_id", "distname", "kpi_id", "start_hour").agg(
    f.avg("kpi_value").alias("kpi_mean"),
    f.min("kpi_value").alias("kpi_min"),
    f.max("kpi_value").alias("kpi_max"),
    f.count("*").alias("kpi_count")
    # add other aggregations in need
)

# sample df
kpi_list = pm_df_all.select("kpi_id").distinct().rdd.flatMap(lambda x: x).collect()
kpi_fractions = {k: 0.01 for k in kpi_list}
pm_sample = pm_df_all.sampleBy("kpi_id", kpi_fractions, seed=42).limit(6_000_000)


# pm grouped by bts and cell
pm_df_grouped_distname = pm_df_grouped_hours = pm_df_all.groupBy("bts_id", "distname", "kpi_id").agg(
    f.avg("kpi_value").alias("kpi_mean"),
    f.min("kpi_value").alias("kpi_min"),
    f.max("kpi_value").alias("kpi_max"),
    f.count("*").alias("kpi_count")
    # add other aggregations in need
)


In [ ]:
# print(pm_df_stats.count())
# print(pm_df_grouped_date.count())
# print(pm_df_grouped_hours.count())
# print(pm_sample.count())
# print(pm_df_grouped_distname.count())

pm_df_stats.write.parquet(str(pm_stats_path / "kpi_stats"), mode="overwrite")
pm_df_grouped_date.write.parquet(str(pm_agg_path / "pm_date_agg"), mode="overwrite", partitionBy="start_date")
pm_df_grouped_hours.write.parquet(str(pm_agg_path / "pm_hour_agg"), mode="overwrite")
pm_sample.write.parquet(str(sample_path / "pm_sample"), mode="overwrite")
pm_df_grouped_distname.write.parquet(str(pm_agg_path / "pm_distname_agg"), mode="overwrite")

